In [1]:
import pandas as pd

In [2]:
ref_df = pd.read_csv("country_list.csv")
df = pd.read_csv("data_stunting.csv")

In [3]:
df = df[[
    'SpatialDimValueCode',
    'Period',
    'Dim1',
    'FactValueNumeric'
]]

In [4]:
df = df.rename(columns={
    'SpatialDimValueCode': 'Country Code',
    'Period': 'Year',
    'FactValueNumeric': 'Value'
})

In [5]:
df = df[df['Dim1'] == 'Both sexes']


In [6]:
df['Year'] = df['Year'].astype(int)

In [8]:
years = list(range(2000, 2024))

full_index = pd.MultiIndex.from_product(
    [ref_df['Country Code'], years],
    names=['Country Code', 'Year']
)

full_df = pd.DataFrame(index=full_index).reset_index()

In [9]:
merged = pd.merge(full_df, df[['Country Code', 'Year', 'Value']], 
                  on=['Country Code', 'Year'], how='left')

# Add country names
merged = pd.merge(ref_df, merged, on='Country Code', how='left')

In [10]:
missing = merged[merged['Value'].isnull()]

In [11]:
missing_report = missing.groupby(
    ['Country Name', 'Country Code']
)['Year'].agg(list).reset_index()

missing_report['Missing Count'] = missing_report['Year'].apply(len)

missing_report = missing_report[['Country Name', 'Country Code', 'Missing Count', 'Year']]

missing_report = missing_report.rename(columns={'Year': 'Missing Years'})

In [12]:
missing_report.to_csv("stunting_missing_report.csv", index=False)

print("✅ Missing values report created!")

✅ Missing values report created!


Filling Missing Values with World Bank Dataset 

In [15]:
missing_df = pd.read_csv("stunting_missing_report.csv")
wb_df = pd.read_csv("(Target Variable-01)- Stunting_World_Bank.csv")

In [16]:
years = [str(year) for year in range(2000, 2024)]